# SmartLens-Indic scene text detection training

This notebook uses only the prepared scene manifests built from `BSTD_v17.57.json`. It does not modify the source ZIP archives or split manifests. Training and evaluation have not been run from this notebook.

The official architecture is DB with PPLCNetV3 (scale 0.75), RSEFPN, and DBHead. Its official pretrained backbone checkpoint is `PPLCNetV3_x0_75_ocr_det.pdparams` (verified HTTP size: 20,103,839 bytes / about 19.2 MiB). No model is downloaded during notebook preparation.

This notebook selects official PaddlePaddle 3.3.0 GPU wheels. The current Windows app environment is PaddlePaddle 3.0.0 CPU / PaddleOCR 3.7.0 / PaddleX 3.7.2. The Colab smoke test is not a substitute for loading the copied model in that exact Windows environment.

## Before running

Reuse the same two original ZIP archives already copied once to `MyDrive/SmartLens-Indic/Data/` for the recognition notebook; do not upload duplicate archives. For this notebook, copy `dataset_work/detection/train.jsonl`, `validation.jsonl`, and `test.jsonl` into `MyDrive/SmartLens-Indic/dataset_work/detection/`, and also copy `excluded_no_polygons.jsonl`. Keep the ZIPs unchanged. Colab cannot read Windows `C:\` paths; the notebook extracts selected images to temporary `/content/` only and stores logs/checkpoints in Drive.

Expected split counts are 867 / 109 / 238 images. Four images with no polygons were excluded rather than treated as verified negative examples. The official 238-image test partition remains independent.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/SmartLens-Indic')
ARCHIVE_DIR = PROJECT / 'Data'
MANIFEST_DIR = PROJECT / 'dataset_work' / 'detection'
ARCHIVES = {path.name: path for path in ARCHIVE_DIR.glob('Smartlens Dataset-*.zip')}
assert len(ARCHIVES) == 2, f'Expected both original ZIPs under {ARCHIVE_DIR}; found {len(ARCHIVES)}'
for name in ('train.jsonl', 'validation.jsonl', 'test.jsonl', 'excluded_no_polygons.jsonl'):
    assert (MANIFEST_DIR / name).is_file(), f'Missing {MANIFEST_DIR / name}'
print('Drive inputs found:', *ARCHIVES.keys(), sep='\n- ')
print('Original ZIPs will be opened read-only; extraction target is /content only.')

In [ ]:
# Detect the Colab runtime before installation. Paddle's official Linux wheel docs list Python 3.9-3.13.
import platform, re, subprocess, sys
assert (3, 9) <= sys.version_info[:2] <= (3, 13), f'Unsupported Python: {platform.python_version()}'
gpu_info = subprocess.run(['nvidia-smi'], check=True, text=True, capture_output=True).stdout
print(gpu_info.splitlines()[0])
match = re.search(r'CUDA Version:\s*(\d+\.\d+)', gpu_info)
assert match, 'Could not detect the driver CUDA version; stop and inspect the Colab runtime.'
cuda_version = float(match.group(1))
if cuda_version >= 13.0: wheel = 'cu130'
elif cuda_version >= 12.9: wheel = 'cu129'
elif cuda_version >= 12.6: wheel = 'cu126'
elif cuda_version >= 11.8: wheel = 'cu118'
else: raise RuntimeError(f'No verified Paddle 3.3.0 wheel selected for CUDA {cuda_version}')
print('Selected official Paddle wheel index:', wheel)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'paddlepaddle-gpu==3.3.0', '-i', f'https://www.paddlepaddle.org.cn/packages/stable/{wheel}/'], check=True)
import paddle
paddle.set_device('gpu:0')
assert paddle.is_compiled_with_cuda(), 'Installed Paddle is CPU-only; stop before training.'
assert paddle.device.get_device().startswith('gpu'), paddle.device.get_device()
print('Python:', platform.python_version(), '| Paddle:', paddle.__version__, '| Device:', paddle.device.get_device())

In [ ]:
# Pin the official training source and install its declared training requirements.
import subprocess
PPOCR_ROOT = Path('/content/PaddleOCR')
PPOCR_COMMIT = 'b03f46425e8ff4442b268ce449e3eef758146cd4'  # PaddleOCR v3.7.0
if not PPOCR_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PaddlePaddle/PaddleOCR.git', str(PPOCR_ROOT)], check=True)
subprocess.run(['git', '-C', str(PPOCR_ROOT), 'checkout', PPOCR_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PPOCR_ROOT / 'requirements.txt')], check=True)
print('PaddleOCR source revision:', subprocess.check_output(['git', '-C', str(PPOCR_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Load exact prepared splits and check group/hash leakage before opening image members.
import json, zipfile, cv2, numpy as np, hashlib, math
EXPECTED = {'train': 867, 'validation': 109, 'test': 238}
rows_by_split, group_owner, image_owner = {}, {}, {}
for split, expected_count in EXPECTED.items():
    rows = [json.loads(line) for line in (MANIFEST_DIR / f'{split}.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    assert len(rows) == expected_count, f'{split}: expected {expected_count}, got {len(rows)}'
    rows_by_split[split] = rows
    for row in rows:
        assert row['split'] == split and row['annotation_archive_member'].endswith('/BSTD_v17.57.json')
        for key, owners in (('group_id', group_owner), ('sha256', image_owner)):
            previous = owners.setdefault(row[key], split)
            assert previous == split, f'{key} leaks across splits: {row[key]}'
excluded = [json.loads(line) for line in (MANIFEST_DIR / 'excluded_no_polygons.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(excluded) == 4 and all('no polygon' in row['reason'] for row in excluded)
print('Split counts and group/hash separation verified; excluded images:', len(excluded))

In [ ]:
# Read-only ZIP access; validate every polygon against the decoded image and generate PaddleOCR label TSVs.
TRAINING_DATA_VALIDATED = False
import os
WORK = Path('/content/smartlens_detection')
DET_DATA = WORK / 'train_data'
LIST_DIR = WORK / 'labels'
DET_DATA.mkdir(parents=True, exist_ok=True)
LIST_DIR.mkdir(parents=True, exist_ok=True)
zip_handles = {name: zipfile.ZipFile(path, 'r') for name, path in ARCHIVES.items()}
list_paths, bad_hindi_labels, polygon_total, geometry_issues = {}, 0, 0, []
try:
    for split, rows in rows_by_split.items():
        label_path = LIST_DIR / f'{split}.txt'
        with label_path.open('w', encoding='utf-8', newline='') as output:
            for row in rows:
                archive_name = Path(row['archive_path'].replace('\\', '/')).name
                assert archive_name in zip_handles, f'Archive not transferred: {archive_name}'
                member = row['archive_member']
                raw = zip_handles[archive_name].read(member)
                assert hashlib.sha256(raw).hexdigest() == row['sha256'], f'Image bytes do not match manifest hash: {member}'
                image = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
                assert image is not None and image.size, f'Unreadable scene image: {member}'
                height, width = image.shape[:2]
                name = hashlib.sha256(member.encode('utf-8')).hexdigest()[:20] + Path(member).suffix.lower()
                target = DET_DATA / 'images' / split / name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(raw)
                paddle_annotations = []
                for annotation_index, annotation in enumerate(row['annotations']):
                    points = annotation['coordinates']
                    assert len(points) >= 3, f'Polygon has fewer than 3 points: {row["image_name"]}'
                    assert all(len(point) == 2 and all(math.isfinite(float(value)) for value in point) for point in points), row['image_name']
                    outside_points = [
                    {'point_index': point_index, 'coordinates': [x, y]}
                    for point_index, (x, y) in enumerate(points)
                    if not (0 <= float(x) <= width and 0 <= float(y) <= height)
                    ]
                    if outside_points:
                        geometry_issues.append({'split': split, 'image_name': row['image_name'], 'archive_path': row['archive_path'], 'archive_member': member, 'annotation_archive_member': row['annotation_archive_member'], 'image_width': width, 'image_height': height, 'annotation_index': annotation_index, 'text': annotation.get('text', ''), 'script_language': annotation.get('script_language', ''), 'original_coordinates': points, 'invalid_coordinates': outside_points})
                    text = str(annotation.get('text', ''))
                    if str(annotation.get('script_language', '')).lower() == 'hindi' and not any('\u0900' <= ch <= '\u097f' for ch in text):
                        bad_hindi_labels += 1
                    paddle_annotations.append({'transcription': text, 'points': points})
                    polygon_total += 1
                relative = target.relative_to(DET_DATA).as_posix()
                output.write(relative + '\t' + json.dumps(paddle_annotations, ensure_ascii=False) + '\n')
        list_paths[split] = label_path
finally:
    for handle in zip_handles.values(): handle.close()
if geometry_issues:
    report_path = PROJECT / 'training' / 'detection' / 'annotation_geometry_issues.jsonl'
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(''.join(json.dumps(issue, ensure_ascii=False) + '\n' for issue in geometry_issues), encoding='utf-8')
    raise ValueError(f'Found {len(geometry_issues)} out-of-image polygons. Training stopped without clipping or changing annotations; review {report_path} and correct a separate working manifest before retrying.')
assert polygon_total > 0
print('Polygons validated:', polygon_total, '| Hindi-tagged annotations without Devanagari:', bad_hindi_labels)
print('Those 129 text-label issues do not invalidate detection polygons; detection uses coordinates, not Hindi transcription quality.')
print('Working images and PaddleOCR label lists are under', WORK)
TRAINING_DATA_VALIDATED = True

The official `DetLabelEncode` expects a TSV image path followed by JSON objects with `transcription` and `points`. It supports variable polygon vertex counts by padding the final point; the prepared annotations vary from 4 to 90 vertices. This notebook preserves the original vertices and stops on non-finite, too-short, or out-of-image polygons instead of clipping them silently. The 129 Hindi-tagged polygons without Devanagari remain valid geometry for detection; they are not recognition ground truth.

In [ ]:
# Update a copy of the exact official config. Keep DB/PPLCNetV3/RSEFPN/DBHead and pretrained checkpoint.
import yaml
OFFICIAL_CONFIG = PPOCR_ROOT / 'configs/det/PP-OCRv5/PP-OCRv5_mobile_det.yml'
TRAIN_CONFIG = WORK / 'ppocrv5_mobile_det_training.yml'
CHECKPOINT_DIR = PROJECT / 'training' / 'detection' / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
config = yaml.safe_load(OFFICIAL_CONFIG.read_text(encoding='utf-8'))
assert config['Architecture']['algorithm'] == 'DB' and config['Architecture']['Backbone']['name'] == 'PPLCNetV3'
config['Global']['save_model_dir'] = str(CHECKPOINT_DIR)
config['Global']['save_epoch_step'] = 1
config['Global']['checkpoints'] = None
config['Global']['pretrained_model'] = 'https://paddleocr.bj.bcebos.com/pretrained/PPLCNetV3_x0_75_ocr_det.pdparams'
config['Train']['dataset']['data_dir'] = str(DET_DATA)
config['Train']['dataset']['label_file_list'] = [str(list_paths['train'])]
config['Eval']['dataset']['data_dir'] = str(DET_DATA)
config['Eval']['dataset']['label_file_list'] = [str(list_paths['validation'])]
TRAIN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('Training config copy:', TRAIN_CONFIG)
print('Official pretrained checkpoint is about 19.2 MiB; PaddleOCR downloads it only when the training cell is run.')

## Training (expensive; run only after all checks above pass)

The training config saves complete `latest` checkpoints to Drive after every epoch. The training cell requires successful dataset and geometry validation. After a Colab interruption, remount Drive and rerun the setup and validation cells; the training cell automatically resumes only when `latest.pdparams`, `latest.pdopt`, and `latest.states` are all present. It stops for invalid polygons or an incomplete checkpoint rather than silently clipping annotations or starting over. Console logs are appended in the same Drive checkpoint folder. This cell is intentionally not run during notebook preparation.

In [ ]:
assert TRAINING_DATA_VALIDATED is True, 'Run and pass dataset and geometry validation before training.'
assert set(list_paths) == {'train', 'validation', 'test'}
assert all(path.is_file() for path in list_paths.values()), 'One or more validated label lists are missing.'
assert not geometry_issues, 'Out-of-image polygons were found; inspect the saved report before training.'
checkpoint_prefix = CHECKPOINT_DIR / 'latest'
checkpoint_files = [Path(str(checkpoint_prefix) + suffix) for suffix in ('.pdparams', '.pdopt', '.states')]
existing_checkpoints = [checkpoint_file.is_file() for checkpoint_file in checkpoint_files]
if any(existing_checkpoints) and not all(existing_checkpoints):
    missing = [str(file) for file, exists in zip(checkpoint_files, existing_checkpoints) if not exists]
    raise RuntimeError(f'Incomplete latest checkpoint; missing {missing}. Preserve the Drive checkpoint folder and inspect it before continuing.')
RESUME = all(existing_checkpoints)
if RESUME:
    print('Resuming from complete Drive checkpoint:', checkpoint_prefix)
else:
    print('No complete latest checkpoint found; this is a fresh training run.')
command = [sys.executable, 'tools/train.py', '-c', str(TRAIN_CONFIG)]
if RESUME:
    command += ['-o', f'Global.checkpoints={checkpoint_prefix}']
log_path = CHECKPOINT_DIR / 'colab_train_console.log'
with log_path.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=PPOCR_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = process.wait()
if code:
    raise RuntimeError(f'Training exited with code {code}; inspect {log_path} before resuming.')


In [ ]:
# Official validation/test DetMetric evaluation against the prepared polygons.
TEST_CONFIG = WORK / 'ppocrv5_mobile_det_test.yml'
test_config = yaml.safe_load(TRAIN_CONFIG.read_text(encoding='utf-8'))
test_config['Eval']['dataset']['label_file_list'] = [str(list_paths['test'])]
TEST_CONFIG.write_text(yaml.safe_dump(test_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
checkpoint = CHECKPOINT_DIR / 'best_accuracy'
for split, cfg in (('validation', TRAIN_CONFIG), ('test', TEST_CONFIG)):
    command = [sys.executable, 'tools/eval.py', '-c', str(cfg), '-o', f'Global.checkpoints={checkpoint}']
    result = subprocess.run(command, cwd=PPOCR_ROOT, text=True, capture_output=True)
    log = PROJECT / 'training' / 'detection' / f'{split}_official_eval.log'
    log.write_text(result.stdout + '\n' + result.stderr, encoding='utf-8')
    print(result.stdout[-5000:])
    result.check_returncode()
    print('Full log:', log)
print('Reported metric is official DetMetric hmean against this polygon test split, not real-world scene accuracy.')

## Export and model-layout compatibility check

The exporter writes inference weights/configuration. The following cell packages them into the `TextDetection(model_name=..., model_dir=...)` layout used by the Windows application and runs a CPU smoke test. Then copy the model to Windows and repeat the same smoke test with PaddlePaddle 3.0.0 CPU / PaddleOCR 3.7.0 / PaddleX 3.7.2 before adoption. A Colab smoke test covers only Colab's runtime. Model loadability does not prove detector accuracy.

In [ ]:
import shutil
EXPORT_ROOT = PROJECT / 'training' / 'detection' / 'export'
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_CONFIG = WORK / 'ppocrv5_mobile_det_export.yml'
export_config = yaml.safe_load(TRAIN_CONFIG.read_text(encoding='utf-8'))
export_config['Global']['checkpoints'] = str(checkpoint)
export_config['Global']['pretrained_model'] = None
export_config['Global']['save_inference_dir'] = str(EXPORT_ROOT)
EXPORT_CONFIG.write_text(yaml.safe_dump(export_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
subprocess.run([sys.executable, 'tools/export_model.py', '-c', str(EXPORT_CONFIG)], cwd=PPOCR_ROOT, check=True)
WINDOWS_MODEL = PROJECT / 'training' / 'detection' / 'windows_model'
WINDOWS_MODEL.mkdir(parents=True, exist_ok=True)
for file in (EXPORT_ROOT / 'inference').iterdir():
    if file.is_file(): shutil.copy2(file, WINDOWS_MODEL / file.name)
shutil.copy2(EXPORT_ROOT / 'inference.yml', WINDOWS_MODEL / 'inference.yml')
assert (WINDOWS_MODEL / 'inference.yml').is_file()
assert (WINDOWS_MODEL / 'inference.pdiparams').is_file(), 'Exported Paddle parameters are missing.'
assert (WINDOWS_MODEL / 'inference.json').is_file() or (WINDOWS_MODEL / 'inference.pdmodel').is_file(), 'Exported Paddle graph is missing.'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'paddleocr==3.7.0'], check=True)
from paddleocr import TextDetection
sample_row = rows_by_split['test'][0]
sample_name = hashlib.sha256(sample_row['archive_member'].encode('utf-8')).hexdigest()[:20] + Path(sample_row['archive_member']).suffix.lower()
sample_path = DET_DATA / 'images' / 'test' / sample_name
predictor = TextDetection(model_name='PP-OCRv5_mobile_det', model_dir=str(WINDOWS_MODEL), device='cpu')
prediction = predictor.predict(input=str(sample_path))
assert prediction, 'No detection result returned from exported model.'
print('CPU export smoke test result:', prediction[0].json)
print('Packaged model directory:', WINDOWS_MODEL)

## Evaluation boundaries

The official `DetMetric` hmean is reported only against the prepared independent BSTD_v17.57 polygon test split. It is not a measure of end-to-end Hindi recognition, spoken-text correctness, or performance on arbitrary real-world photographs. Keep these detection results separate from the IndicSTR12 recognition results. Do not claim training or inference results until the relevant cells have actually completed.